# How Does AI Adoption Rate Compare with Productivity Change?
### ENGE707: Data Engineering and Machine Learning Pipeline Project — Phase 1 Scoping Report

Suemon Kwok (14883335), Quentin Masoe (23191960), Avathanshu Bhat (23227963)

**Dataset:** Global AI Adoption and Workforce Impact Dataset (`ai_company_adoption.csv`), Kaggle, CC0 - 150,000 quarterly company records, 43 columns, Q1 2023 to Q4 2026.
Link: https://www.kaggle.com/datasets/mohankrishnathalla/global-ai-adoption-and-workforce-impact-dataset

**Research question:** how does a company's AI adoption rate compare with the productivity change it subsequently reports?

**Scope note.** An earlier draft of this project looked at several AI-related variables at once (maturity, ethics governance, workforce displacement, investment). Following lecturer feedback, the project was narrowed to a single comparison so each pipeline stage could be examined in depth:

| Role | Column | Meaning |
|---|---|---|
| Focus variable 1 | `ai_adoption_rate` | % of a company's operations/processes that use AI |
| Focus variable 2 | `productivity_change_percent` | % change in productivity subsequently reported |

This notebook covers the four Phase 1 tasks: problem and dataset selection, data acquisition and inspection, data cleansing and transformation, and exploratory data analysis.

In [ ]:
import os
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

## Task 1 — Problem Definition and Dataset Selection

- **Source:** Global AI Adoption and Workforce Impact Dataset (Kaggle, CC0), provided as `ai_company_adoption.csv`.
- **Structure:** tabular, alphanumeric, 150,000 rows x 43 columns — satisfies the assignment's requirement of more than 20 and fewer than 300 features.
- **Unit of observation:** one row = one company's survey response for one quarter (a company can appear multiple times, once per quarter, identified by `company_id`).
- **Stakeholders:** executives deciding AI budgets, technology investment committees, and policymakers (e.g. the OECD) who need evidence of whether AI adoption is converting into measurable productivity gains.

## Task 2 — Data Acquisition, Inspection and Documentation

### 2.1 Loading and structural inspection

In [ ]:
# Load the dataset (extract from archive.zip first if the CSV isn't already alongside this notebook)
CSV_NAME = "ai_company_adoption.csv"
ZIP_NAME = "archive.zip"

if not os.path.exists(CSV_NAME) and os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")

df = pd.read_csv(CSV_NAME)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# quarter takes only four repeating values, so it is converted to a category dtype;
# the other categorical columns sit outside the narrowed two-variable comparison
df["quarter"] = df["quarter"].astype("category")

print("Column dtype counts:")
print(df.dtypes.value_counts())

print("\nColumns spanning:", df["country"].nunique(), "countries and",
      df["industry"].nunique(), "industries")
print("Company size categories:", df["company_size"].unique().tolist())

The dataset spans 30 countries and 9 industries and separates companies into three size categories (Startup, SME, Enterprise). This categorical variety is not used directly in the narrowed two-variable comparison, but it confirms the dataset covers a broad population of companies rather than a narrow or unrepresentative sample.

In [ ]:
df_original = df.copy(deep=True)

print("Records:", df_original.shape[0])
print("Columns:", df_original.shape[1])

print("\nFirst look at the two focus columns:")
print(df_original[["ai_adoption_rate", "productivity_change_percent"]].describe())

### 2.2 Data-quality inspection of the focus variables

In [ ]:
focus = ["ai_adoption_rate", "productivity_change_percent"]

# Missing values
missing_focus = df_original[focus].isna().sum()
print("Missing values in the focus columns:")
print(missing_focus)
if missing_focus.sum() == 0:
    print("-> Confirmed: no missing data in either focus column.")

# Duplicate rows / duplicate response IDs
print("\nNumber of fully duplicate rows:", df_original.duplicated().sum())
print("Number of duplicate response_id values:", df_original["response_id"].duplicated().sum())

# Value ranges
print("\nMinimum and maximum values:")
print(df_original[focus].agg(["min", "max"]).T)

**Findings:** neither `ai_adoption_rate` nor `productivity_change_percent` has a missing value across all 150,000 records, and there are no fully duplicate rows or duplicate `response_id` values. `ai_adoption_rate` runs 0-100% and `productivity_change_percent` runs 0-34.36%, with no negative values, so no implausible or out-of-range values needed correction at this stage.

### 2.3 Assumption documented during inspection

Because each `company_id` repeats across quarters, with 10,000 companies each responding an unequal number of times, the dataset is treated as an unbalanced panel rather than as 150,000 fully independent observations. This is revisited later as a limitation.

## Task 3 — Data Cleansing and Transformation

### 3.1 Cleaning decisions

Because Task 2 found no missing values, duplicates, or invalid records in the focus variables, `clean_data()` works on a deep copy of `df_original`, removes any record with a missing or out-of-range focus value, removes duplicates, and narrows the working table to the two focus variables plus three identifier columns (`company_id`, `survey_year`, `quarter`) kept for traceability. This narrowing is treated as being as important a cleaning decision as removing an invalid value, as it is the direct response to the lecturer's Phase 1 scope feedback.

In [ ]:
def clean_data(data):
    clean = data.copy(deep=True)

    # Safeguard: drop any record with a missing or out-of-range focus value
    invalid_record = (
        clean["ai_adoption_rate"].isna()
        | clean["productivity_change_percent"].isna()
        | ~clean["ai_adoption_rate"].between(0, 100)
    )
    print("Invalid/missing records removed:", invalid_record.sum())

    clean = clean.loc[~invalid_record].copy()
    clean = clean.drop_duplicates()

    # Narrow the working frame down to the single comparison, plus lightweight identifiers
    keep_cols = ["company_id", "survey_year", "quarter",
                 "ai_adoption_rate", "productivity_change_percent"]
    clean = clean[keep_cols].reset_index(drop=True)
    return clean


df_clean = clean_data(df_original)

print("\nOriginal shape:", df_original.shape)
print("Cleaned shape:", df_clean.shape)
print("Rows removed:", len(df_original) - len(df_clean))
print("Duplicate rows remaining:", df_clean.duplicated().sum())

### 3.2 Outlier investigation (interquartile range method)

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    below = (series < lower).sum()
    above = (series > upper).sum()
    return below, above, lower, upper


for col in focus:
    below, above, lower, upper = iqr_outliers(df_clean[col])
    pct = (below + above) / len(df_clean) * 100
    print(f"{col}: {below} below lower fence, {above} above upper fence "
          f"({pct:.2f}% of {len(df_clean)} records) | fence = [{lower:.2f}, {upper:.2f}]")

Both variables' outliers sit only above the upper fence and are well under 1% of records. Consistent with the course's distinction between a statistical outlier and a data-entry error, these values are retained rather than removed: they fall inside each variable's valid documented range and represent genuinely high-adoption or high-productivity companies. Removing them would weaken the relationship this project was narrowed to examine.

### 3.3 ETL and ELT demonstration

## Task 4 — Exploratory Data Analysis and Visualisation

### 4.1 Descriptive statistics

In [ ]:
desc = df_clean[focus].describe().round(2)
skew = df_clean[focus].skew().round(2)
print(desc)
print("\nSkewness:")
print(skew)

`ai_adoption_rate` has a mean of 36.41% and standard deviation of 14.54%, with a skew of 0.06 (close to symmetric). `productivity_change_percent` has a mean of 9.27% and standard deviation of 5.64%, with a skew of 0.30 (mild right-skew), read alongside the visualisations below rather than as a conclusion on their own.

### 4.2 Distribution analysis

In [ ]:
plt.hist(df_clean["ai_adoption_rate"], bins=20, color="steelblue")
plt.xlabel("AI adoption rate (%)")
plt.ylabel("Number of company-quarters")
plt.title("Figure 2. Distribution of AI adoption rate")
plt.show()

plt.hist(df_clean["productivity_change_percent"], bins=20, color="darkorange")
plt.xlabel("Productivity change (%)")
plt.ylabel("Number of company-quarters")
plt.title("Figure 3. Distribution of productivity change")
plt.show()

`ai_adoption_rate` (Figure 2) is close to symmetric, peaking between 30% and 40%, with very few companies near 0% or above 80%. `productivity_change_percent` (Figure 3) rises to a peak between 8% and 10% before tailing off to the right, matching its positive skew, with a secondary rise near 0-2% investigated further under Relationship Analysis.

### 4.3 Outlier analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].boxplot(df_clean["ai_adoption_rate"], tick_labels=["AI adoption rate"])
axes[0].set_ylabel("%")
axes[1].boxplot(df_clean["productivity_change_percent"], tick_labels=["Productivity change"])
axes[1].set_ylabel("%")
plt.suptitle("Figure 4. Boxplots of AI adoption rate and productivity change")
plt.show()

The IQR for `ai_adoption_rate` is narrower than the IQR for `productivity_change_percent`, indicating higher variability in productivity change. Both panels show points beyond the whiskers, consistent with the IQR outlier counts in Task 3 — a boxplot identifies statistical rarity rather than proving that a value is incorrect.

### 4.4 Relationship analysis

In [ ]:
x = df_clean["ai_adoption_rate"]
y = df_clean["productivity_change_percent"]

# Sample for plotting only (150,000 points is too dense to read); all statistics below use the full data
sample = df_clean.sample(n=3000, random_state=42)

plt.scatter(sample["ai_adoption_rate"], sample["productivity_change_percent"], alpha=0.4, s=10)

slope, intercept = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
plt.plot(x_line, slope * x_line + intercept, color="red", linewidth=2, label="Fitted trend")

plt.xlabel("AI adoption rate (%)")
plt.ylabel("Productivity change (%)")
plt.title("Figure 5. AI adoption rate vs productivity change")
plt.legend()
plt.show()

pearson_r = x.corr(y)
print(f"Pearson correlation coefficient: {pearson_r:.3f}")
print(f"Fitted line: productivity_change_percent = {slope:.4f} * ai_adoption_rate + {intercept:.2f}")

The point cloud rises from bottom-left to top-right, and the fitted trend line confirms a positive, roughly linear relationship. The Pearson correlation coefficient is 0.675, a moderately strong positive linear association. The spread of points around the trend line shows adoption rate does not fully determine productivity change on its own — correlation does not equal causation.

**Zero-productivity subgroup:**

In [ ]:
zero_prod = df_clean[df_clean["productivity_change_percent"] == 0]
print("Records with exactly 0% productivity change:", len(zero_prod),
      f"({len(zero_prod) / len(df_clean) * 100:.1f}% of the cleaned dataset)")
print("\nAI adoption rate within this zero-productivity subgroup:")
print(zero_prod["ai_adoption_rate"].describe().round(2))
print("\nAI adoption rate across the full cleaned dataset (for comparison):")
print(df_clean["ai_adoption_rate"].describe().round(2))

The zero-productivity subgroup (over 9,000 records) is not spread across the full adoption-rate range: its adoption rate tops out at roughly 61%, against a full-sample maximum of 100%, and its mean adoption rate (about 18%) is roughly half the full-sample mean (about 36%). Reporting zero productivity change is concentrated among lower-adoption companies specifically, rather than scattered evenly across the adoption scale.

### 4.5 Grouped comparison

In [ ]:
bins = [-0.01, 33.3, 66.6, 100]
labels = ["Low", "Medium", "High"]
adoption_band = pd.cut(df_clean["ai_adoption_rate"], bins=bins, labels=labels)

band_means = df_clean.groupby(adoption_band, observed=True)["productivity_change_percent"].mean()
print(band_means.round(2))

plt.bar(band_means.index.astype(str), band_means.values, color=["#a6cee3", "#1f78b4", "#08306b"])
plt.xlabel("AI adoption band")
plt.ylabel("Mean productivity change (%)")
plt.title("Figure 6. Mean productivity change by AI adoption band")
plt.show()

Mean productivity change increases step-wise from the Low band to the Medium and High bands. This grouped view is a derived summary of the same two focus variables rather than a new variable, and is read together with the scatter plot in Figure 5 and boxplots in Figure 4 rather than as evidence on its own.

### 4.6 Correlation heatmap

In [ ]:
corr_matrix = df_clean[focus].corr()
print(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(focus)))
ax.set_yticks(range(len(focus)))
ax.set_xticklabels(focus, rotation=45, ha="right")
ax.set_yticklabels(focus)
for i in range(len(focus)):
    for j in range(len(focus)):
        ax.text(j, i, f"{corr_matrix.values[i, j]:.2f}", ha="center", va="center")
ax.set_title("Figure 7. Correlation heatmap")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

The heatmap confirms the Pearson correlation coefficient reported above — a second, visual confirmation of the relationship already shown in the scatter plot. A pair plot and missing-value plot were considered but excluded: with only two focus variables, a pair plot would repeat Figures 2, 3 and 5, and a missing-value plot would show only two bars at zero height given the confirmed absence of missing data in Task 2.